# Core PCE Inflation Shock Momentum (ISM) Index

This notebook adapts Lansing and Shapiro (2026), *Measuring Inflation Shock Momentum*, from the 129-category headline PCE basket to the 117-category core PCE basket in `core_lines.xlsx`. It constructs the index, produces a Figure-1-style chart, and exports index and category-level CSV files.

The baseline uses category monthly inflation, 120-month rolling AR(1) models with a constant, and the signs of the last three in-sample residuals from the single window ending in month $t$. Current nominal PCE spending weights aggregate the category signals.

**Deliberate deviations from the paper:** (1) the basket is core PCE rather than headline PCE; (2) if category signals or spending are missing, weights are dynamically renormalized over the available core categories, subject to the minimum-line setting below. The history is reconstructed using the current BEA vintage and is not a real-time-vintage backtest.

## 1. Settings and imports

All method and output settings are collected here. The local BEA credential is used only in API requests and is never printed or exported.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd().resolve()
CORE_LINES_PATH = ROOT / "core_lines.xlsx"
CACHE = ROOT / "cache"
OUT = ROOT / "output" / "ism_core_pce"
CACHE.mkdir(exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Local credential: do not print, log, or include it in exported files.
BEA_API_KEY = "FD4E7F46-5D89-4CA8-BE48-4F5997F83B3B"
BEA_DATASET = "NIUnderlyingDetail"
BEA_FREQUENCY = "M"
PRICE_TABLE = "U20404"
SPEND_TABLE = "U20405"
START_YEAR = 1959
END_YEAR = datetime.now().year
YEAR_CHUNKS = [(1959, 1999), (2000, END_YEAR)]

WINDOW = 120
AR_LAGS = 1
K_RUN = 3
MIN_LINES = 50
REFRESH = False
TAG = f"k{K_RUN}_ar{AR_LAGS}"

INDEX_CSV = OUT / f"ism_index_{TAG}.csv"
FLAGS_CSV = OUT / f"momentum_flags_{TAG}.csv"
FIGURE_PNG = OUT / f"ism_figure1_{TAG}.png"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)
warnings.filterwarnings("default")

try:
    display
except NameError:
    display = print

print({"window": WINDOW, "ar_lags": AR_LAGS, "k_run": K_RUN, "min_lines": MIN_LINES, "refresh": REFRESH})

## 2. BEA data helpers

The helpers fetch monthly BEA Underlying Detail data in two year chunks, retry transient failures, validate the response shape, and cache normalized long-form tables. Cached files are used by default.

In [ ]:
BEA_URL = "https://apps.bea.gov/api/data"
MISSING_VALUES = {"", ".", "-", "--", "NA", "(NA)", "N/A", "nan", "None"}

def parse_bea_month(value):
    text = str(value).strip()
    match = re.fullmatch(r"(\d{4})M(\d{1,2})", text)
    if not match:
        raise ValueError(f"Unexpected BEA monthly period: {value!r}")
    return pd.Timestamp(int(match.group(1)), int(match.group(2)), 1)

def as_float(value):
    text = str(value).replace(",", "").strip()
    return np.nan if text in MISSING_VALUES else float(text)

def unwrap_results(raw):
    results = raw.get("BEAAPI", {}).get("Results", {})
    if isinstance(results, list):
        candidates = [item for item in results if isinstance(item, dict) and ("Data" in item or "Error" in item)]
        results = candidates[-1] if candidates else {}
    if not isinstance(results, dict):
        raise RuntimeError("Unexpected BEA Results payload")
    if "Error" in results:
        error = results["Error"]
        if isinstance(error, list):
            error = error[0] if error else {}
        code = error.get("APIErrorCode", "unknown") if isinstance(error, dict) else "unknown"
        message = error.get("APIErrorDescription", str(error)) if isinstance(error, dict) else str(error)
        raise RuntimeError(f"BEA API error {code}: {message}")
    return results

def normalize_bea_rows(rows):
    out = []
    for row in rows or []:
        line = row.get("LineNumber")
        period = row.get("TimePeriod")
        if line is None or period is None:
            continue
        out.append({
            "line": int(str(line).strip()),
            "description": str(row.get("LineDescription", "")).strip(),
            "series_code": str(row.get("SeriesCode", "")).strip(),
            "date": parse_bea_month(period),
            "value": as_float(row.get("DataValue")),
        })
    return pd.DataFrame(out, columns=["line", "description", "series_code", "date", "value"])

def fetch_bea(table, year0, year1, retries=4):
    params = {
        "UserID": BEA_API_KEY,
        "method": "GetData",
        "DataSetName": BEA_DATASET,
        "TableName": table,
        "Frequency": BEA_FREQUENCY,
        "Year": ",".join(str(year) for year in range(year0, year1 + 1)),
        "ResultFormat": "JSON",
    }
    body = urlencode(params).encode("utf-8")
    last_error = None
    for attempt in range(retries):
        try:
            request = Request(BEA_URL, data=body, headers={"User-Agent": "core-pce-ism-replication/1.0"})
            with urlopen(request, timeout=180) as response:
                raw = json.loads(response.read().decode("utf-8"))
            results = unwrap_results(raw)
            data = normalize_bea_rows(results.get("Data", []))
            if data.empty:
                raise RuntimeError(f"BEA returned no data for {table}, {year0}-{year1}")
            return data
        except (HTTPError, URLError, TimeoutError, RuntimeError, ValueError, json.JSONDecodeError) as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(20 * (attempt + 1))
    raise RuntimeError(f"Could not fetch {table}, {year0}-{year1}: {last_error}")

def cache_path(table):
    return CACHE / f"bea_{BEA_DATASET}_{table}_{BEA_FREQUENCY}_{START_YEAR}_{END_YEAR}.csv"

def bea_table(table, refresh=REFRESH):
    path = cache_path(table)
    if path.exists() and not refresh:
        cached = pd.read_csv(path, parse_dates=["date"])
        required = {"line", "description", "series_code", "date", "value"}
        if required.issubset(cached.columns):
            return cached.sort_values(["line", "date"]).reset_index(drop=True)
        warnings.warn(f"Ignoring malformed cache: {path}")
    chunks = []
    for chunk_no, (year0, year1) in enumerate(YEAR_CHUNKS):
        chunks.append(fetch_bea(table, year0, year1))
        if chunk_no < len(YEAR_CHUNKS) - 1:
            time.sleep(2)
    data = pd.concat(chunks, ignore_index=True)
    duplicates = data.duplicated(["line", "date"], keep=False)
    if duplicates.any():
        sample = data.loc[duplicates, ["line", "date"]].head().to_dict("records")
        raise AssertionError(f"Duplicate BEA line/date rows in {table}: {sample}")
    data = data.sort_values(["line", "date"]).reset_index(drop=True)
    data.to_csv(path, index=False, date_format="%Y-%m-%d")
    return data

def pivot_selected(data, lines, table):
    selected = data[data["line"].isin(lines)].copy()
    if selected.duplicated(["line", "date"]).any():
        raise AssertionError(f"Duplicate selected line/date rows in {table}")
    wide = selected.pivot(index="date", columns="line", values="value").sort_index()
    return wide.reindex(columns=lines)

## 3. Core basket and BEA coverage

`core_lines.xlsx` is the basket source of truth. The checks below require 117 unique line numbers, verify their presence in both BEA tables, compare descriptions, and summarize time-series coverage.

In [ ]:
raw_lines = pd.read_excel(CORE_LINES_PATH)
required_columns = {"Line Item", "Description"}
if not required_columns.issubset(raw_lines.columns):
    raise ValueError(f"{CORE_LINES_PATH.name} must contain {sorted(required_columns)}")

core_lines = raw_lines.rename(columns={"Line Item": "line", "Description": "description", "Code": "code"}).copy()
core_lines["line"] = core_lines["line"].astype(int)
core_lines["description"] = core_lines["description"].astype(str).str.strip()
if "code" not in core_lines:
    core_lines["code"] = ""
core_lines["code"] = core_lines["code"].fillna("").astype(str).str.strip()
core_lines = core_lines[["line", "description", "code"]].sort_values("line").reset_index(drop=True)

assert len(core_lines) == 117, f"Expected 117 core lines, found {len(core_lines)}"
assert core_lines["line"].is_unique, "Core line numbers must be unique"
assert core_lines["description"].is_unique, "Core descriptions must be unique"
LINES = core_lines["line"].tolist()
LINE_DESCRIPTION = core_lines.set_index("line")["description"]
print(f"Loaded {len(LINES)} unique core PCE line items.")
display(core_lines.head(12))

In [ ]:
run_started = time.perf_counter()
price_long = bea_table(PRICE_TABLE)
spend_long = bea_table(SPEND_TABLE)

for table_name, data in [(PRICE_TABLE, price_long), (SPEND_TABLE, spend_long)]:
    if data.duplicated(["line", "date"]).any():
        raise AssertionError(f"{table_name} contains duplicate line/date observations")
    missing_lines = sorted(set(LINES) - set(data["line"].unique()))
    if missing_lines:
        raise AssertionError(f"{table_name} is missing configured lines: {missing_lines}")

prices = pivot_selected(price_long, LINES, PRICE_TABLE)
spend = pivot_selected(spend_long, LINES, SPEND_TABLE)
full_dates = pd.date_range(min(prices.index.min(), spend.index.min()), max(prices.index.max(), spend.index.max()), freq="MS")
prices = prices.reindex(full_dates).where(lambda frame: frame > 0)
spend = spend.reindex(full_dates)

def normalized_description(value):
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

description_checks = []
for table_name, data in [(PRICE_TABLE, price_long), (SPEND_TABLE, spend_long)]:
    bea_desc = data[data["line"].isin(LINES)].drop_duplicates("line").set_index("line")["description"]
    for line in LINES:
        expected = LINE_DESCRIPTION.loc[line]
        observed = bea_desc.get(line, "")
        if normalized_description(expected) != normalized_description(observed):
            description_checks.append({"table": table_name, "line": line, "workbook": expected, "bea": observed})
if description_checks:
    warnings.warn(f"{len(description_checks)} workbook/BEA description mismatches; inspect `description_mismatches`.")
description_mismatches = pd.DataFrame(description_checks)
if not description_mismatches.empty:
    display(description_mismatches)

def interior_gaps(series):
    valid = series.dropna()
    return 0 if valid.empty else int(series.loc[valid.index.min():valid.index.max()].isna().sum())

coverage = pd.DataFrame({
    "description": LINE_DESCRIPTION,
    "price_first": prices.apply(lambda s: s.first_valid_index()),
    "price_last": prices.apply(lambda s: s.last_valid_index()),
    "price_interior_gaps": prices.apply(interior_gaps),
    "spend_first": spend.apply(lambda s: s.first_valid_index()),
    "spend_last": spend.apply(lambda s: s.last_valid_index()),
    "spend_interior_gaps": spend.apply(interior_gaps),
}).reindex(LINES)
nonpositive_mask = spend.notna() & spend.le(0)
nonpositive_spend = int(nonpositive_mask.sum().sum())
nonpositive_spend_observations = (
    spend.where(nonpositive_mask).stack().dropna().rename("spending").reset_index()
    .rename(columns={"level_0": "date", "line": "line"})
)
if nonpositive_spend:
    warnings.warn(f"Found {nonpositive_spend:,} nonpositive category-spending observations; they will not receive weights.")
    display(nonpositive_spend_observations)
print(f"BEA data span: {full_dates.min():%Y-%m} to {full_dates.max():%Y-%m}")
print(f"Interior gaps — prices: {coverage['price_interior_gaps'].sum():,}; spending: {coverage['spend_interior_gaps'].sum():,}")
display(coverage.head(12))

## 4. Monthly inflation and rolling autoregressions

Monthly category inflation is the simple percentage change in each U20404 price index. The paper does not explicitly distinguish simple from log percentage changes; the simple change is used here, and multiplication by a positive annualization factor would not alter residual signs.

For every end month $t$, a single AR($p$) is estimated on the 120-month window ending at $t$. The returned residual block is ordered $(t,t-1,\ldots,t-k+1)$, so January 1969 can be classified from the final three residuals of the first complete window.

In [ ]:
infl = prices.pct_change(fill_method=None) * 100.0

def rolling_last_residuals(y, window=WINDOW, p=AR_LAGS, k=K_RUN):
    """Return recent residuals and coefficients from each rolling AR(p).

    residuals[e, j] is the in-sample residual at month e-j from the
    single `window`-month regression ending at e. Coefficients are
    ordered as the constant followed by lags 1 through p.
    """
    y = np.asarray(y, dtype=float)
    if window <= p or k < 1 or k > window - p:
        raise ValueError("Require window > p and 1 <= k <= window-p")
    residuals_out = np.full((len(y), k), np.nan)
    coefficients_out = np.full((len(y), p + 1), np.nan)
    for end in range(window - 1, len(y)):
        sample = y[end - window + 1:end + 1]
        if np.isnan(sample).any():
            continue
        dependent = sample[p:]
        design = np.column_stack(
            [np.ones(window - p)] + [sample[p - lag:window - lag] for lag in range(1, p + 1)]
        )
        beta = np.linalg.lstsq(design, dependent, rcond=None)[0]
        residual = dependent - design @ beta
        residuals_out[end, :] = residual[-k:][::-1]
        coefficients_out[end, :] = beta
    return residuals_out, coefficients_out

resid_end = pd.DataFrame(np.nan, index=infl.index, columns=LINES)
intercept = pd.DataFrame(np.nan, index=infl.index, columns=LINES)
rho1 = pd.DataFrame(np.nan, index=infl.index, columns=LINES)
m_pos = pd.DataFrame(False, index=infl.index, columns=LINES)
m_neg = pd.DataFrame(False, index=infl.index, columns=LINES)
valid = pd.DataFrame(False, index=infl.index, columns=LINES)
residual_blocks = {}

for line in LINES:
    block, coefficients = rolling_last_residuals(infl[line].to_numpy())
    residual_blocks[line] = block
    block_valid = ~np.isnan(block).any(axis=1)
    resid_end[line] = block[:, 0]
    intercept[line] = coefficients[:, 0]
    rho1[line] = coefficients[:, 1]
    valid[line] = block_valid
    m_pos[line] = block_valid & (block > 0).all(axis=1)
    m_neg[line] = block_valid & (block < 0).all(axis=1)

first_valid_by_line = valid.apply(lambda s: s.index[s.to_numpy().argmax()] if s.any() else pd.NaT)
full_history_lines = coverage.index[coverage["price_first"].eq(pd.Timestamp(START_YEAR, 1, 1))].tolist()
if full_history_lines:
    first_full_history_dates = first_valid_by_line.loc[full_history_lines]
    if first_full_history_dates.isna().any():
        raise AssertionError(f"No valid classification for full-history lines: {first_full_history_dates[first_full_history_dates.isna()].index.tolist()}")
    unexpected_first_dates = first_full_history_dates[first_full_history_dates.ne(pd.Timestamp(1969, 1, 1))]
    if not unexpected_first_dates.empty:
        raise AssertionError(f"Unexpected first classifications: {unexpected_first_dates.to_dict()}")
    first_full_history = pd.Timestamp(1969, 1, 1)
else:
    warnings.warn("No line has a January 1959 first price observation; January 1969 timing could not be asserted.")

# Independent AR(1) check using np.polyfit for the first valid line/window.
if AR_LAGS == 1:
    check_line = next(line for line in LINES if valid[line].any())
    check_end = valid.index[valid[check_line].to_numpy().argmax()]
    end_loc = infl.index.get_loc(check_end)
    sample = infl[check_line].iloc[end_loc - WINDOW + 1:end_loc + 1].to_numpy()
    slope, constant = np.polyfit(sample[:-1], sample[1:], 1)
    check_residual = sample[1:] - (constant + slope * sample[:-1])
    np.testing.assert_allclose(residual_blocks[check_line][end_loc], check_residual[-K_RUN:][::-1], atol=1e-10, rtol=0)

print(f"First full-history classification: {first_full_history:%Y-%m}" if full_history_lines else "Timing check unavailable")
print("Final-month rho(1) percentiles:")
display(rho1.iloc[-1].dropna().quantile([0.10, 0.25, 0.50, 0.75, 0.90]).rename("rho1"))

## 5. Current spending weights and the core ISM index

A category receives a current-month weight only when its $k$ residuals are valid and current nominal spending is finite and positive. Weights are renormalized over those categories. This dynamic available-basket treatment is an explicit deviation from the paper, so both the number of weighted categories and their share of total core spending are retained.

In [ ]:
spend = spend.reindex(index=infl.index, columns=LINES)
valid_weight = valid & spend.notna() & spend.gt(0)
weighted_spend = spend.where(valid_weight)
available_spend = weighted_spend.sum(axis=1, min_count=1)
full_core_spend = spend.where(spend.gt(0)).sum(axis=1, min_count=1)
weights = weighted_spend.div(available_spend, axis=0)
n_valid = valid_weight.sum(axis=1).astype(int)
weight_coverage = available_spend.div(full_core_spend)
eligible = n_valid.ge(MIN_LINES) & available_spend.gt(0)

S_plus = (weights * m_pos.astype(float)).sum(axis=1, min_count=1).where(eligible)
S_minus = (weights * m_neg.astype(float)).sum(axis=1, min_count=1).where(eligible)
ISM = (S_plus - S_minus).where(eligible)

published_weight_sum = weights.sum(axis=1, min_count=1).where(eligible).dropna()
np.testing.assert_allclose(published_weight_sum.to_numpy(), 1.0, atol=1e-12, rtol=0)
assert S_plus.dropna().between(0, 1).all()
assert S_minus.dropna().between(0, 1).all()
assert ((S_plus + S_minus).dropna() <= 1 + 1e-12).all()

signal_denominator = valid.stack().sum()
positive_share = float(m_pos.stack().sum() / signal_denominator)
negative_share = float(m_neg.stack().sum() / signal_denominator)
print(f"Unweighted valid line-month shares — M+: {positive_share:.1%}; M-: {negative_share:.1%}")
print(f"Published ISM span: {ISM.first_valid_index():%Y-%m} to {ISM.last_valid_index():%Y-%m}")
print(f"Minimum published line count: {n_valid.loc[ISM.dropna().index].min()} of {len(LINES)}")
print(f"Minimum published spending coverage: {weight_coverage.loc[ISM.dropna().index].min():.2%}")

eras = {
    "Great Inflation (1974-1981)": ("1974-01", "1981-12"),
    "Volcker disinflation (1982-1986)": ("1982-01", "1986-12"),
    "1990s": ("1990-01", "1999-12"),
    "Mid-2000s (2004-2007)": ("2004-01", "2007-12"),
    "Post-Great-Recession (2010-2019)": ("2010-01", "2019-12"),
    "Pandemic surge (2021-2022)": ("2021-01", "2022-12"),
}
era_summary = pd.DataFrame({
    label: {"ISM mean": ISM.loc[start:end].mean(), "S+ mean": S_plus.loc[start:end].mean(), "S- mean": S_minus.loc[start:end].mean()}
    for label, (start, end) in eras.items()
}).T
display(era_summary.round(3))
print("Historical comparisons are descriptive only; core PCE need not reproduce the paper's headline-basket magnitudes.")

## 6. Aggregate core PCE benchmark

The benchmark is the unique non-market-based U20404 series whose description contains “excluding food and energy.” Its 12-month percentage change is used only for the middle chart panel and the index CSV.

In [ ]:
price_series = price_long[["line", "description", "series_code"]].drop_duplicates()
candidate_mask = (
    price_series["description"].str.fullmatch(
        r"(?:PCE|Personal consumption expenditures) excluding food and energy", case=False, na=False
    )
    & ~price_series["description"].str.contains("market-based", case=False, na=False)
)
core_candidates = price_series.loc[candidate_mask].drop_duplicates("line").sort_values("line")
print("Core aggregate candidates:")
display(core_candidates)
if len(core_candidates) != 1:
    raise AssertionError(f"Expected one non-market-based core PCE aggregate, found {len(core_candidates)}")
core_line = int(core_candidates.iloc[0]["line"])
core_description = core_candidates.iloc[0]["description"]
core_price = (
    price_long.loc[price_long["line"].eq(core_line), ["date", "value"]]
    .drop_duplicates("date")
    .set_index("date")["value"]
    .sort_index()
    .reindex(infl.index)
)
core_pce_yoy = core_price.pct_change(12, fill_method=None) * 100.0
print(f"Benchmark line {core_line}: {core_description}")

## 7. Figure-1-style chart

The paper overlays headline PCE inflation and ISM on dual axes. This presentation adaptation uses three aligned panels and no dual axes: net core ISM, 12-month core PCE inflation, and the positive/negative momentum components.

In [ ]:
plot_data = pd.DataFrame({"ISM": ISM, "core_pce_yoy": core_pce_yoy, "S_plus": S_plus, "S_minus": S_minus}).dropna(subset=["ISM"])
ink = "#17212b"
grid = "#d8dee6"
ism_color = "#1f5a93"
positive_color = "#d95f02"
negative_color = "#2c7fb8"
benchmark_color = "#4f5965"

fig, axes = plt.subplots(3, 1, figsize=(13.0, 9.2), sharex=True, gridspec_kw={"height_ratios": [1.15, 0.9, 1.0], "hspace": 0.20})

axes[0].plot(plot_data.index, plot_data["ISM"], color=ism_color, lw=1.35)
axes[0].axhline(0, color=ink, lw=0.8)
axes[0].set_ylabel("Net share")
axes[0].set_title("A. Core PCE Inflation Shock Momentum Index", loc="left", fontsize=11.5, fontweight="bold")

axes[1].plot(plot_data.index, plot_data["core_pce_yoy"], color=benchmark_color, lw=1.35)
axes[1].axhline(2, color=grid, lw=0.9, ls="--")
axes[1].set_ylabel("Percent")
axes[1].set_title("B. Core PCE inflation (12-month)", loc="left", fontsize=11.5, fontweight="bold")

axes[2].plot(plot_data.index, plot_data["S_plus"], color=positive_color, lw=1.25, label="Positive momentum share (S+)")
axes[2].plot(plot_data.index, plot_data["S_minus"], color=negative_color, lw=1.25, label="Negative momentum share (S-)")
axes[2].set_ylabel("Share")
axes[2].set_title("C. Components of the core ISM index", loc="left", fontsize=11.5, fontweight="bold")
axes[2].legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False, fontsize=9.2)

for axis in axes:
    axis.grid(axis="y", color=grid, lw=0.7)
    axis.spines[["top", "right", "left"]].set_visible(False)
    axis.spines["bottom"].set_color(grid)
    axis.tick_params(axis="both", colors=ink, labelsize=9)
    axis.margins(x=0)

axes[2].xaxis.set_major_locator(mdates.YearLocator(8))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
last_date = plot_data.index[-1]
axes[2].set_xlim(plot_data.index[0], last_date + pd.DateOffset(months=20))
for column, color, offset in [("S_plus", positive_color, 6), ("S_minus", negative_color, -10)]:
    axes[2].annotate(
        f"{column.replace('_', ' ')}  {plot_data[column].iloc[-1]:.2f}",
        xy=(last_date, plot_data[column].iloc[-1]),
        xytext=(8, offset), textcoords="offset points",
        color=color, fontsize=8.5, va="center",
    )

fig.suptitle("Core PCE Inflation Shock Momentum", x=0.085, y=0.985, ha="left", fontsize=17, fontweight="bold", color=ink)
fig.text(0.085, 0.952, f"117-category core basket · {WINDOW}-month rolling AR({AR_LAGS}) · {K_RUN} same-signed residuals", ha="left", fontsize=10, color=benchmark_color)
fig.text(0.085, 0.012, "Source: BEA NIPA Underlying Detail tables U20404 and U20405; author calculations. Current-vintage historical reconstruction.", ha="left", fontsize=8.4, color=benchmark_color)
fig.subplots_adjust(left=0.085, right=0.97, top=0.91, bottom=0.12)
fig.savefig(FIGURE_PNG, dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print(FIGURE_PNG)

## 8. CSV exports

The index file contains one row per published month. The long file contains one row per valid weighted category-month and retains the endpoint residual, sign flags, normalized weight, and first AR coefficient for diagnostics.

In [ ]:
index_output = pd.DataFrame({
    "date": ISM.index,
    "S_plus": S_plus.to_numpy(),
    "S_minus": S_minus.to_numpy(),
    "ISM": ISM.to_numpy(),
    "n_valid": n_valid.to_numpy(),
    "weight_coverage": weight_coverage.to_numpy(),
    "core_pce_yoy": core_pce_yoy.to_numpy(),
}).dropna(subset=["ISM"]).reset_index(drop=True)
index_output.to_csv(INDEX_CSV, index=False, date_format="%Y-%m", float_format="%.10g")

flag_parts = []
for line in LINES:
    line_frame = pd.DataFrame({
        "date": infl.index,
        "line": line,
        "description": LINE_DESCRIPTION.loc[line],
        "resid_end": resid_end[line].to_numpy(),
        "M_plus": m_pos[line].astype(np.int8).to_numpy(),
        "M_minus": m_neg[line].astype(np.int8).to_numpy(),
        "weight": weights[line].to_numpy(),
        "rho1": rho1[line].to_numpy(),
    })
    line_frame = line_frame.loc[valid_weight[line].to_numpy()]
    flag_parts.append(line_frame)
flags_output = pd.concat(flag_parts, ignore_index=True).sort_values(["date", "line"]).reset_index(drop=True)
flags_output.to_csv(FLAGS_CSV, index=False, date_format="%Y-%m", float_format="%.10g")

print(f"Wrote {len(index_output):,} index rows to {INDEX_CSV}")
print(f"Wrote {len(flags_output):,} valid category-month rows to {FLAGS_CSV}")
display(index_output.tail())

## 9. Robustness settings and caveats

To reproduce the paper's principal robustness variants, change `K_RUN` to 2 or 4 or change `AR_LAGS` to 3 or 12 in the settings cell and run all cells. The output tag prevents those files from overwriting the baseline.

Exact values need not match the headline index because this notebook uses core PCE, allows a dynamically available basket, and uses the current BEA vintage. Historical episode signs and the paper's approximate 20%/15% positive/negative observation shares are descriptive diagnostics rather than acceptance criteria. COVID-era months remain in the sample.

In [ ]:
runtime_seconds = time.perf_counter() - run_started
summary = pd.Series({
    "core_line_count": len(LINES),
    "bea_data_start": full_dates.min().strftime("%Y-%m"),
    "bea_data_end": full_dates.max().strftime("%Y-%m"),
    "first_classification": ISM.first_valid_index().strftime("%Y-%m"),
    "last_classification": ISM.last_valid_index().strftime("%Y-%m"),
    "window_months": WINDOW,
    "ar_lags": AR_LAGS,
    "same_sign_run": K_RUN,
    "minimum_lines": MIN_LINES,
    "latest_ISM": float(ISM.dropna().iloc[-1]),
    "latest_weight_coverage": float(weight_coverage.loc[ISM.last_valid_index()]),
    "description_mismatch_count": len(description_mismatches),
    "nonpositive_spending_observations": nonpositive_spend,
    "runtime_seconds": round(runtime_seconds, 2),
    "index_csv": str(INDEX_CSV),
    "flags_csv": str(FLAGS_CSV),
    "figure_png": str(FIGURE_PNG),
})
display(summary)